In [ ]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd


In [ ]:
# Source - https://stackoverflow.com/a/51014127
# Posted by M. Deckers, modified by community. See post 'Timeline' for change history
# Retrieved 2026-04-03, License - CC BY-SA 4.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
df_tv = pd.read_csv('data/train.csv')


In [ ]:
continous_variables = df_tv.select_dtypes(['float64']).columns
from sklearn.preprocessing import LabelEncoder
class_le = LabelEncoder()
y = class_le.fit_transform(df_tv['Irrigation_Need'].values)

In [ ]:
df_dummy = pd.get_dummies(df_tv.iloc[:,1:-1], dtype=int)
from sklearn.preprocessing import StandardScaler

#df_dummy[continous_variables] = StandardScaler().fit_transform(df_dummy[continous_variables])

#x = df_dummy.to_numpy()

sc = StandardScaler().fit(df_dummy[continous_variables])
df_dummy[continous_variables] = sc.transform(df_dummy[continous_variables])
x = df_dummy.to_numpy()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = \
    train_test_split(x, y, 
                     test_size=0.20,
                     stratify=y,
                     random_state=1)

In [ ]:
from torch.utils.data import DataLoader, TensorDataset


X_train = torch.tensor(X_train).float()
y_train = torch.tensor(y_train).long()
X_train.to(device)
y_train.to(device)
train_ds = TensorDataset(X_train, y_train)

batch_size = 8
torch.manual_seed(1)
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

In [ ]:
hidden_units = [32, 16]
input_size = X_train.shape[1]

all_layers = [nn.Flatten()]
for hidden_unit in hidden_units:
    layer = nn.Linear(input_size, hidden_unit)
    all_layers.append(layer)
    all_layers.append(nn.ReLU())
    input_size = hidden_unit

all_layers.append(nn.Linear(hidden_units[-1], 3))
model = nn.Sequential(*all_layers)
model.to(device)

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

torch.manual_seed(1)
num_epochs = 20
for epoch in range(num_epochs):
    accuracy_hist_train = 0
    for x_batch, y_batch in train_dl:
        pred = model(x_batch)
        loss = loss_fn(pred, y_batch)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        is_correct = (torch.argmax(pred, dim=1) == y_batch).float()
        accuracy_hist_train += is_correct.sum()
    accuracy_hist_train /= len(train_dl.dataset)
    print(f'Epoch {epoch}  Accuracy {accuracy_hist_train:.4f}')

In [ ]:
pred = model(X_test)
is_correct = (torch.argmax(pred, dim=1) == y_test).float()
print(f'Test accuracy: {is_correct.mean():.4f}') 

In [ ]:
df_test = pd.read_csv('data/test.csv')

ids = df_test['id'].values


df_test_dummy = pd.get_dummies(df_test.iloc[:,1:])
df_test_dummy[continous_variables] = sc.transform(df_test_dummy[continous_variables])
x_test = df_test_dummy.to_numpy()

In [ ]:
df_submission_gs = pd.DataFrame({'id': ids, 'Irrigation_Need': class_le.inverse_transform(model(x_test))})
df_submission_gs.to_csv('data/submission-nn_v1.csv', index=False)